In [0]:
%sql
DESCRIBE TABLE cursodataacademy.proyecto_hotel.hotel_bookings;

Pregunta 1. CREATE / INSERT – Creación de tabla de prueba
Enunciado: El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_reservas_prueba
USING DELTA
AS
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE 1 = 0;

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_reservas_prueba
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE hotel = 'City Hotel'
AND is_canceled = 1;

In [0]:
%sql
SELECT COUNT(*) AS total_reservas_canceladas
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba;

conclusion:Se creó correctamente la tabla tb_reservas_prueba, incorporando 33,102 registros correspondientes a reservas canceladas del City Hotel. Esta información permite separar las cancelaciones para realizar un análisis específico y apoyar la toma de decisiones comerciales.

Pregunta 2. UPDATE – Corrección de datos
Enunciado: Se detectó que los huéspedes que registraron bebés (babies) en realidad no tenían bebés. Actualiza la tabla de prueba y cambia ese valor a 0 para todos los registros.

In [0]:
%sql
UPDATE cursodataacademy.proyecto_hotel.tb_reservas_prueba
SET babies = 0;

Conclusion:La actualización permite corregir los registros de bebés en la tabla de prueba, estableciendo el valor en cero para todas las reservas. Esto evita utilizar información incorrecta en los análisis posteriores.

Pregunta 3. CAST / CONCAT – Formato de fecha
Enunciado: Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").

In [0]:
%sql
SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT(
        LPAD(CAST(arrival_date_day_of_month AS STRING), 2, '0'),
        '-',
        arrival_date_month,
        '-',
        CAST(arrival_date_year AS STRING)
    ) AS arrival_date_string
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

conclusion: Se generó la columna arrival_date_string con las fechas de llegada en un formato uniforme y legible, facilitando el análisis temporal y la identificación de las reservas.

Pregunta 4. LIKE – Filtrado de canales de distribución
Enunciado: Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.

In [0]:
%sql
SELECT *
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel LIKE 'T%';

conclusion: Se identificaron las reservas cuyos canales de distribución comienzan con la letra T, permitiendo analizar la participación de las agencias de viajes y otros canales relacionados en la captación de reservas.

Pregunta 5. COUNT / SUM / AVG – Resumen general de reservas
Enunciado: Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).

In [0]:
%sql
SELECT
    COUNT(*) AS total_reservas,
    SUM(total_of_special_requests) AS total_solicitudes_especiales,
    AVG(adr) AS promedio_precio_noche
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

conclusion: Se obtuvo un resumen general de las reservas, las solicitudes especiales y el precio promedio por noche. Estos indicadores permiten conocer el volumen de operaciones y las características generales de la demanda hotelera.

Pregunta 6. DELETE – Eliminación de reservas de prueba
Enunciado: En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.

In [0]:
%sql
DELETE FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
WHERE adr = 0.0;

In [0]:
%sql
SELECT COUNT(*) AS registros_con_adr_cero
FROM cursodataacademy.proyecto_hotel.tb_reservas_prueba
WHERE adr = 0.0;

conclusion: Se eliminaron de la tabla de prueba las reservas con precio por noche igual a cero, evitando que registros considerados pruebas del sistema afecten el análisis comercial.

Pregunta 7. GROUP BY – Reservas por canal de distribución
Enunciado: El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.

In [0]:
%sql
SELECT
    distribution_channel,
    COUNT(*) AS cantidad_reservas
FROM cursodataacademy.proyecto_hotel.hotel_bookings
GROUP BY distribution_channel
ORDER BY cantidad_reservas DESC;

conclusion: Se identificó la cantidad de reservas por cada canal de distribución, permitiendo reconocer cuáles generan mayor volumen de operaciones y evaluar su importancia para la estrategia comercial del hotel.

Pregunta 8. CASE WHEN – Clasificación de reservas por precio
Enunciado: Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium".

In [0]:
%sql
SELECT
    adr,
    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        WHEN adr > 150 THEN 'Premium'
        ELSE 'Sin clasificación'
    END AS categoria_precio
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

conclusion: Se clasificaron las reservas según su precio por noche en tres categorías: Económico, Estándar y Premium. Esta segmentación facilita el análisis de precios y la identificación de los distintos segmentos de clientes.

Pregunta 9. DATE_SUB / DATE_ADD – Cálculo de fecha de reserva
Enunciado: Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio. Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.

In [0]:
%sql
SELECT
    lead_time,
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    DATE_SUB(
        TO_DATE(
            CONCAT(
                CAST(arrival_date_year AS STRING), '-',
                arrival_date_month, '-',
                CAST(arrival_date_day_of_month AS STRING)
            ),
            'yyyy-MMMM-d'
        ),
        CAST(lead_time AS INT)
    ) AS booking_date
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

conclusion: Se calculó la fecha de realización de cada reserva restando los días de anticipación a la fecha de llegada. Este resultado permite conocer cuándo se efectuaron las reservaciones y analizar su comportamiento temporal.

Pregunta 10. Funciones matemáticas – Cálculo de anticipación
Enunciado: Queremos calcular una "cuota de riesgo". Selecciona el lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada meses_anticipacion_techo.

In [0]:
%sql
SELECT
    lead_time,
    CEIL(lead_time / 30.0) AS meses_anticipacion_techo
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

conclusion: Se calculó el tiempo de anticipación de las reservas en meses aproximados, aplicando redondeo hacia arriba. Este indicador permite identificar con cuánta anticipación los clientes realizan sus reservaciones.